# MeterMind — Model 2: GRU Seq2Seq with Attention

This notebook implements the **learned neural baseline** for MeterMind.

A GRU encoder-decoder with Bahdanau attention learns to compress expanded archaic prose back into metrically structured lines — purely from examples, no explicit stress rules.

### How it works
```
INPUT  : 'Thou feedest thy light own flame with fuel that is self substantial'
ENCODER: reads each word left-to-right, producing a hidden state at each step
ATTENTION: at each decoder step, learns which encoder states to focus on
DECODER: generates one word at a time, attending to the right input words
OUTPUT : 'feedest thy lights flame with self substantial fuel'
```

### Why attention?
Without attention, the encoder compresses the whole input into a single vector — a bottleneck. With attention, the decoder can look back at *all* encoder states at each step, learning which input word belongs at each output position. This maps naturally onto the reordering task.

### Pipeline
```
training_pairs.csv → vocab → GRU+attention → train → evaluate (MA, SP, G)
```

## 0. Install dependencies

In [ ]:
%pip install pronouncing sentence-transformers torch --quiet

## 1. Imports & configuration

In [ ]:
import re, io, csv, math, json, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from google.colab import files
from sentence_transformers import SentenceTransformer, util
import pronouncing

# ── Reproducibility ───────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# ── Device ────────────────────────────────────────────────────────────────────
# Automatically uses GPU if available (Colab provides a free T4)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {DEVICE}')

# ── Hyperparameters ───────────────────────────────────────────────────────────
MAX_LEN    = 30     # max sequence length (words)
EMBED_DIM  = 128    # word embedding size
HIDDEN_DIM = 256    # GRU hidden state size
BATCH_SIZE = 32
EPOCHS     = 50
LR         = 0.001
CLIP       = 1.0    # gradient clipping to prevent exploding gradients
TEACHER_FORCING_RATIO = 0.5  # probability of using ground truth token during training

print('Config ready.')

## 2. Load dataset

In [ ]:
uploaded = files.upload()
filename = list(uploaded.keys())[0]
df = pd.read_csv(io.StringIO(uploaded[filename].decode('utf-8')))

pairs = list(zip(df['input'].tolist(), df['target'].tolist()))
random.shuffle(pairs)

# Train / val / test split — split on unique targets so no target leaks across splits
unique_targets = list({t for _, t in pairs})
random.shuffle(unique_targets)
n = len(unique_targets)
train_targets = set(unique_targets[:int(n * 0.8)])
val_targets   = set(unique_targets[int(n * 0.8):int(n * 0.9)])
test_targets  = set(unique_targets[int(n * 0.9):])

train_pairs = [(s, t) for s, t in pairs if t in train_targets]
val_pairs   = [(s, t) for s, t in pairs if t in val_targets]
test_pairs  = [(s, t) for s, t in pairs if t in test_targets]

print(f'Total pairs : {len(pairs):,}')
print(f'Train       : {len(train_pairs):,}')
print(f'Val         : {len(val_pairs):,}')
print(f'Test        : {len(test_pairs):,}')
print(f'\nExample:')
print(f'  INPUT  : {pairs[0][0]}')
print(f'  TARGET : {pairs[0][1]}')

## 3. Vocabulary

We build a word-level vocabulary from all training pairs.
Four special tokens:
- `<PAD>` — padding to make sequences the same length in a batch
- `<SOS>` — start of sequence signal to the decoder
- `<EOS>` — end of sequence, tells decoder to stop
- `<UNK>` — unknown word (fallback for words not seen in training)

In [ ]:
def tokenize(text):
    """Lowercase, strip punctuation, split into word tokens."""
    return re.sub(r"[^a-zA-Z\s]", "", text.lower()).split()


def build_vocab(pairs):
    vocab = {'<PAD>': 0, '<SOS>': 1, '<EOS>': 2, '<UNK>': 3}
    for src, tgt in pairs:
        for word in tokenize(src) + tokenize(tgt):
            if word not in vocab:
                vocab[word] = len(vocab)
    return vocab


def encode(words, vocab, max_len):
    """Convert word list to padded integer sequence with SOS and EOS."""
    ids = [vocab['<SOS>']] + [vocab.get(w, vocab['<UNK>']) for w in words] + [vocab['<EOS>']]
    ids = ids[:max_len]
    return ids + [vocab['<PAD>']] * (max_len - len(ids))


# Build vocab from training pairs only (no data leakage)
vocab        = build_vocab(train_pairs)
idx_to_word  = {v: k for k, v in vocab.items()}
VOCAB_SIZE   = len(vocab)

print(f'Vocabulary size: {VOCAB_SIZE:,}')

## 4. Dataset & DataLoader

In [ ]:
class PoetryDataset(Dataset):
    """
    Each item returns:
      src     : encoded input (expanded prose)
      tgt_in  : encoded target with SOS (decoder input)
      tgt_out : encoded target with EOS (what the decoder should predict)
    """
    def __init__(self, pairs, vocab, max_len):
        self.pairs   = pairs
        self.vocab   = vocab
        self.max_len = max_len

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        src_text, tgt_text = self.pairs[idx]
        src = encode(tokenize(src_text), self.vocab, self.max_len)
        tgt = encode(tokenize(tgt_text), self.vocab, self.max_len)
        src = torch.tensor(src, dtype=torch.long)
        tgt = torch.tensor(tgt, dtype=torch.long)
        return src, tgt[:-1], tgt[1:]   # tgt_in (SOS...), tgt_out (...EOS)


train_loader = DataLoader(PoetryDataset(train_pairs, vocab, MAX_LEN), batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(PoetryDataset(val_pairs,   vocab, MAX_LEN), batch_size=BATCH_SIZE)
test_loader  = DataLoader(PoetryDataset(test_pairs,  vocab, MAX_LEN), batch_size=BATCH_SIZE)

print(f'Train batches : {len(train_loader)}')
print(f'Val batches   : {len(val_loader)}')
print(f'Test batches  : {len(test_loader)}')

## 5. Model architecture

### Encoder
A GRU that reads the input word by word, producing:
- `encoder_outputs` — hidden state at every time step (used by attention)
- `hidden` — final hidden state (used to initialise the decoder)

### Attention (Bahdanau)
At each decoder step, attention computes a score between the current decoder hidden state and every encoder output. These scores are softmaxed into weights, then used to build a **context vector** — a weighted sum of encoder outputs. Intuitively: "given where I am in decoding, which input words should I focus on?"

### Decoder
At each step the decoder takes:
1. The previous output word (or `<SOS>` at the start)
2. The context vector from attention
3. Its own previous hidden state
...and predicts the next output word.

In [ ]:
class Encoder(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.gru       = nn.GRU(embed_dim, hidden_dim, batch_first=True)
        self.dropout   = nn.Dropout(0.3)

    def forward(self, src):
        # src: [batch, seq_len]
        embedded = self.dropout(self.embedding(src))        # [batch, seq_len, embed_dim]
        outputs, hidden = self.gru(embedded)                # outputs: [batch, seq_len, hidden_dim]
        return outputs, hidden


class BahdanauAttention(nn.Module):
    """
    Computes attention weights over encoder outputs.
    score(decoder_hidden, encoder_output) = v * tanh(W1*decoder + W2*encoder)
    """
    def __init__(self, hidden_dim):
        super().__init__()
        self.W1 = nn.Linear(hidden_dim, hidden_dim)
        self.W2 = nn.Linear(hidden_dim, hidden_dim)
        self.v  = nn.Linear(hidden_dim, 1, bias=False)

    def forward(self, decoder_hidden, encoder_outputs):
        # decoder_hidden  : [batch, hidden_dim]
        # encoder_outputs : [batch, src_len, hidden_dim]
        dec = self.W1(decoder_hidden).unsqueeze(1)          # [batch, 1, hidden_dim]
        enc = self.W2(encoder_outputs)                      # [batch, src_len, hidden_dim]
        scores = self.v(torch.tanh(dec + enc)).squeeze(-1)  # [batch, src_len]
        weights = F.softmax(scores, dim=-1)                 # [batch, src_len]
        context = torch.bmm(weights.unsqueeze(1), encoder_outputs).squeeze(1)  # [batch, hidden_dim]
        return context, weights


class Decoder(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.attention = BahdanauAttention(hidden_dim)
        # Input to GRU = embedding + context vector
        self.gru       = nn.GRU(embed_dim + hidden_dim, hidden_dim, batch_first=True)
        self.fc_out    = nn.Linear(hidden_dim, vocab_size)
        self.dropout   = nn.Dropout(0.3)

    def forward(self, tgt_token, hidden, encoder_outputs):
        # tgt_token      : [batch] — single token at this step
        # hidden         : [1, batch, hidden_dim]
        # encoder_outputs: [batch, src_len, hidden_dim]
        embedded = self.dropout(self.embedding(tgt_token.unsqueeze(1)))  # [batch, 1, embed_dim]
        context, attn_weights = self.attention(hidden.squeeze(0), encoder_outputs)
        gru_input = torch.cat([embedded, context.unsqueeze(1)], dim=-1)  # [batch, 1, embed+hidden]
        output, hidden = self.gru(gru_input, hidden)                     # [batch, 1, hidden_dim]
        logits = self.fc_out(output.squeeze(1))                          # [batch, vocab_size]
        return logits, hidden, attn_weights


class Seq2SeqAttention(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim):
        super().__init__()
        self.encoder = Encoder(vocab_size, embed_dim, hidden_dim)
        self.decoder = Decoder(vocab_size, embed_dim, hidden_dim)

    def forward(self, src, tgt, teacher_forcing_ratio=0.5):
        """
        Teacher forcing: during training, feed the ground-truth token as
        decoder input with probability `teacher_forcing_ratio`.
        This speeds up learning by keeping the decoder on track early in training.
        """
        batch_size, tgt_len = tgt.shape
        encoder_outputs, hidden = self.encoder(src)

        # First decoder input is always <SOS>
        dec_input = tgt[:, 0]
        all_logits = []

        for t in range(1, tgt_len):
            logits, hidden, _ = self.decoder(dec_input, hidden, encoder_outputs)
            all_logits.append(logits.unsqueeze(1))
            # Teacher forcing decision
            if random.random() < teacher_forcing_ratio:
                dec_input = tgt[:, t]           # use ground truth
            else:
                dec_input = logits.argmax(-1)   # use model prediction

        return torch.cat(all_logits, dim=1)  # [batch, tgt_len-1, vocab_size]


model     = Seq2SeqAttention(VOCAB_SIZE, EMBED_DIM, HIDDEN_DIM).to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
criterion = nn.CrossEntropyLoss(ignore_index=0)  # ignore PAD tokens in loss

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Model parameters: {n_params:,}')

## 6. Training

In [ ]:
def run_epoch(model, loader, optimizer=None, teacher_forcing_ratio=0.5):
    """Run one epoch. If optimizer is None, runs in eval mode (no gradients)."""
    is_train = optimizer is not None
    model.train() if is_train else model.eval()
    total_loss = 0

    with torch.set_grad_enabled(is_train):
        for src, tgt_in, tgt_out in loader:
            src, tgt_in, tgt_out = src.to(DEVICE), tgt_in.to(DEVICE), tgt_out.to(DEVICE)
            logits = model(src, tgt_in, teacher_forcing_ratio)
            loss   = criterion(logits.reshape(-1, VOCAB_SIZE), tgt_out.reshape(-1))

            if is_train:
                optimizer.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), CLIP)
                optimizer.step()

            total_loss += loss.item()

    return total_loss / len(loader)


train_losses, val_losses = [], []
best_val_loss = float('inf')

for epoch in range(1, EPOCHS + 1):
    train_loss = run_epoch(model, train_loader, optimizer, TEACHER_FORCING_RATIO)
    val_loss   = run_epoch(model, val_loader)

    train_losses.append(train_loss)
    val_losses.append(val_loss)

    # Save best model
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), 'best_model.pt')

    if epoch % 10 == 0:
        print(f'Epoch {epoch:3d}/{EPOCHS}  Train: {train_loss:.4f}  Val: {val_loss:.4f}')

# Plot
plt.figure(figsize=(8, 4))
plt.plot(train_losses, label='Train')
plt.plot(val_losses,   label='Val')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('GRU + Attention — Training Curve')
plt.legend()
plt.tight_layout()
plt.savefig('gru_training_curve.png', dpi=150)
plt.show()
print(f'Best val loss: {best_val_loss:.4f}')

## 7. Inference

Two constraints applied at inference time:
1. **Copy mask** — model can only output words present in the source input (since all target words are guaranteed to be there by dataset construction)
2. **Repetition penalty** — reduces the score of already-used tokens so the model doesn't repeat words

In [ ]:
# Load best checkpoint
model.load_state_dict(torch.load('best_model.pt', map_location=DEVICE))
model.eval()


def get_source_mask(src_ids, vocab_size):
    """Allow only tokens present in the source. Always allow EOS so model can stop."""
    mask = torch.full((vocab_size,), float('-inf'))
    for token_id in src_ids:
        if token_id not in (0, 1, 2, 3):  # skip PAD, SOS, EOS, UNK
            mask[token_id] = 0.0
    mask[vocab['<EOS>']] = 0.0
    return mask.to(DEVICE)


def predict(src_text, max_len=MAX_LEN, repeat_penalty=2.0):
    src_ids  = encode(tokenize(src_text), vocab, max_len)
    src      = torch.tensor(src_ids, dtype=torch.long).unsqueeze(0).to(DEVICE)
    src_mask = get_source_mask(src_ids, VOCAB_SIZE)

    with torch.no_grad():
        encoder_outputs, hidden = model.encoder(src)

    dec_input  = torch.tensor([vocab['<SOS>']], dtype=torch.long).to(DEVICE)
    output_words = []
    used_tokens  = set()

    for _ in range(max_len):
        with torch.no_grad():
            logits, hidden, _ = model.decoder(dec_input, hidden, encoder_outputs)

        logits = logits.squeeze(0) + src_mask
        for tok in used_tokens:
            logits[tok] -= repeat_penalty

        next_token = logits.argmax().item()
        if next_token == vocab['<EOS>']:
            break

        word = idx_to_word.get(next_token, '<UNK>')
        output_words.append(word)
        used_tokens.add(next_token)
        dec_input = torch.tensor([next_token], dtype=torch.long).to(DEVICE)

    return ' '.join(output_words)


# Quick sample
print('Sample predictions:\n')
for src, tgt in test_pairs[:3]:
    out = predict(src)
    print(f'INPUT  : {src}')
    print(f'TARGET : {tgt}')
    print(f'OUTPUT : {out}')
    print()

## 8. Evaluation metrics

In [ ]:
# ── Shared metric utilities ───────────────────────────────────────────────────
# Upload meter_utils.py then eval_metrics.py when prompted.
from google.colab import files as _fu
print('Please upload meter_utils.py')
_fu.upload()
from meter_utils import *
print('Please upload eval_metrics.py')
_fu.upload()
from eval_metrics import *

# Trigger model downloads before the eval loop
load_sp_model()
load_gpt2()
print('All metrics ready.')


## 9. Evaluate on test set

In [ ]:
from tqdm.notebook import tqdm

results = []

for src, tgt in tqdm(test_pairs, desc='Evaluating'):
    output = predict(src)
    words  = tokenize(output)

    ma = metrical_accuracy(words)
    sp = semantic_preservation(' '.join(tokenize(src)), output)
    g  = grammaticality(output)

    results.append({
        'input':  src,
        'target': tgt,
        'output': output,
        'MA':     round(ma, 4),
        'SP':     round(sp, 4),
        'G':      round(g,  4),
    })

results_df = pd.DataFrame(results)
print(f'Evaluated {len(results_df):,} test pairs.')

## 10. Results

In [ ]:
print('=== GRU + Attention — Test Set Results ===')
print(f"Metrical Accuracy (MA)    : {results_df['MA'].mean():.4f} ± {results_df['MA'].std():.4f}")
print(f"Semantic Preservation (SP): {results_df['SP'].mean():.4f} ± {results_df['SP'].std():.4f}")
print(f"Grammaticality (G)        : {results_df['G'].mean():.4f} ± {results_df['G'].std():.4f}")

print('\n=== Sample outputs ===')
for _, row in results_df.head(5).iterrows():
    print(f"INPUT  : {row['input']}")
    print(f"TARGET : {row['target']}")
    print(f"OUTPUT : {row['output']}")
    print(f"MA={row['MA']:.3f}  SP={row['SP']:.3f}  G={row['G']:.3f}")
    print()

# Distribution plots
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
fig.suptitle('GRU + Attention — Metric Distributions', fontsize=14)
for ax, metric, colour in zip(axes, ['MA', 'SP', 'G'], ['steelblue', 'seagreen', 'tomato']):
    ax.hist(results_df[metric], bins=20, color=colour, edgecolor='white', alpha=0.85)
    ax.axvline(results_df[metric].mean(), color='black', linestyle='--', linewidth=1.5,
               label=f"mean={results_df[metric].mean():.3f}")
    ax.set_title(metric)
    ax.set_xlabel('Score')
    ax.set_ylabel('Count')
    ax.legend()
plt.tight_layout()
plt.savefig('gru_attention_metrics.png', dpi=150)
plt.show()

## 11. Save results

In [ ]:
results_df.to_csv('gru_attention_results.csv', index=False)
print('Saved gru_attention_results.csv')

summary = {
    'model': 'GRU + Attention',
    'n':     len(results_df),
    'MA':    {'mean': round(float(results_df['MA'].mean()), 4), 'std': round(float(results_df['MA'].std()), 4)},
    'SP':    {'mean': round(float(results_df['SP'].mean()), 4), 'std': round(float(results_df['SP'].std()), 4)},
    'G':     {'mean': round(float(results_df['G'].mean()),  4), 'std': round(float(results_df['G'].std()),  4)},
}
with open('gru_attention_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)
print('Saved gru_attention_summary.json')

files.download('gru_attention_results.csv')
files.download('gru_attention_summary.json')
files.download('gru_training_curve.png')
files.download('gru_attention_metrics.png')